# Capítulo 12 — Fourier: ver el mundo en frecuencias

**Cuaderno interactivo de *La servilleta y el ordenador*.**

Cada sección reproduce una figura del capítulo. La gracia no es ejecutarlas: es **cambiar los parámetros y comprobar si ocurre lo que esperabas**.

> Antes de ejecutar cada celda, escribe en una línea qué esperas ver. Después mira si ocurrió. Y después, por qué.


In [ ]:
import sys, pathlib
sys.path.insert(0, str(pathlib.Path.cwd() / ''))
sys.path.insert(0, '../../../herramientas')
sys.path.insert(0, str(pathlib.Path.cwd().parents[1] / 'herramientas'))

import numpy as np
import matplotlib.pyplot as plt
from estilo_libro import C, use_style, rng, save

use_style()
%matplotlib inline

---

## Muestreo: ¿por qué las ruedas de las películas giran al revés?

Una sinusoide muestreada por encima y por debajo de Nyquist, y el espectro
resultante con la frecuencia replegada.

La figura responde: ¿qué información se pierde exactamente al muestrear, y se
puede recuperar?

Ejecutar:  python fig_aliasing.py

*(script original: `codigo/fig_aliasing.py`)*

**Antes de ejecutar, escribe aquí qué esperas ver:**

> 


In [ ]:
import pathlib
import sys

import matplotlib.pyplot as plt
import numpy as np

from estilo_libro import C, save, use_style  # noqa: E402

use_style()

FS = 10.0                     # frecuencia de muestreo, Hz
T = 2.0
t_fino = np.linspace(0, T, 4000)
t_muestra = np.arange(0, T, 1 / FS)

fig, axes = plt.subplots(2, 2, figsize=(10.4, 5.6),
                         gridspec_kw={"hspace": 0.45})

for fila, f0 in enumerate([3.0, 8.0]):
    ax = axes[fila, 0]
    ax.plot(t_fino, np.sin(2 * np.pi * f0 * t_fino), color=C.blue, lw=1.2,
            label=f"señal real, {f0:.0f} Hz")
    ax.plot(t_muestra, np.sin(2 * np.pi * f0 * t_muestra), "o", color=C.red,
            ms=6, label=f"muestras a {FS:.0f} Hz")
    f_alias = abs(f0 - FS * round(f0 / FS))
    ax.plot(t_fino, np.sin(2 * np.pi * f_alias * t_fino) *
            np.sign(np.cos(np.pi * round(f0 / FS))), "--", color=C.ochre,
            lw=1.6, label=f"alias, {f_alias:.0f} Hz")
    ax.set_xlim(0, 1), ax.set_ylim(-1.6, 1.6)
    ax.set_xlabel("tiempo (s)")
    ax.set_title(f"$f_0$ = {f0:.0f} Hz  "
                 f"({'por debajo' if f0 < FS/2 else 'POR ENCIMA'} de Nyquist "
                 f"= {FS/2:.0f} Hz)", fontsize=9.5)
    ax.legend(fontsize=7.4, loc="upper right")

    # espectro de las muestras
    ax = axes[fila, 1]
    n = 512
    tt = np.arange(n) / FS
    x = np.sin(2 * np.pi * f0 * tt)
    esp = np.abs(np.fft.rfft(x * np.hanning(n)))
    frec = np.fft.rfftfreq(n, 1 / FS)
    ax.plot(frec, esp / esp.max(), color=C.blue, lw=1.6)
    ax.axvline(FS / 2, color=C.red, ls="--", lw=1.4)
    ax.text(FS / 2 - 0.15, 0.85, "Nyquist", rotation=90, fontsize=8,
            color=C.red, ha="right")
    ax.set_xlabel("frecuencia (Hz)"), ax.set_ylabel("amplitud")
    ax.set_title(f"Lo que ve el analizador: un pico en {f_alias:.0f} Hz",
                 fontsize=9.5)
    print(f"f0={f0} Hz  ->  alias en {f_alias} Hz")

plt.show()

**¿Ocurrió lo que esperabas? ¿Por qué?**

> 

**Juega:** cambia un parámetro de la celda anterior, vuelve a ejecutarla y anota el efecto.

> 


---

## Convolución y filtrado: ¿por qué el teorema de convolución lo cambia todo?

Una señal con ruido, filtrada por convolución en el dominio del tiempo y por
multiplicación en el de la frecuencia, con el coste computacional comparado.

La figura responde: ¿qué es exactamente un filtro?

Ejecutar:  python fig_convolucion.py

*(script original: `codigo/fig_convolucion.py`)*

**Antes de ejecutar, escribe aquí qué esperas ver:**

> 


In [ ]:
import pathlib
import sys

import matplotlib.pyplot as plt
import numpy as np

from estilo_libro import C, rng, save, use_style  # noqa: E402

use_style()
r = rng(12)

N = 2000
t = np.linspace(0, 4, N)
limpia = np.sin(2 * np.pi * 1.5 * t) + 0.6 * np.sin(2 * np.pi * 4.0 * t)
ruido = 0.9 * r.standard_normal(N)
señal = limpia + ruido

# Núcleo gaussiano
ancho = 25
k = np.exp(-0.5 * (np.arange(-3 * ancho, 3 * ancho + 1) / ancho) ** 2)
k /= k.sum()
filtrada = np.convolve(señal, k, mode="same")

fig, axes = plt.subplots(2, 2, figsize=(10.6, 5.6),
                         gridspec_kw={"hspace": 0.45})

ax = axes[0, 0]
ax.plot(t, señal, color=C.grey, lw=0.6, label="con ruido")
ax.plot(t, limpia, color=C.ink, lw=1.6, label="verdad")
ax.set_xlabel("tiempo"), ax.set_title("La señal", fontsize=10)
ax.legend(fontsize=7.6)

ax = axes[0, 1]
ax.plot(np.arange(len(k)) - len(k) // 2, k, color=C.blue, lw=1.8)
ax.set_xlabel("retardo (muestras)")
ax.set_title("El núcleo del filtro $h$", fontsize=10)

ax = axes[1, 0]
ax.plot(t, filtrada, color=C.red, lw=1.6, label="filtrada")
ax.plot(t, limpia, color=C.ink, lw=1.2, alpha=0.7, label="verdad")
ax.set_xlabel("tiempo")
ax.set_title("Convolución: promedio con memoria", fontsize=10)
ax.legend(fontsize=7.6)

ax = axes[1, 1]
frec = np.fft.rfftfreq(N, t[1] - t[0])
H = np.abs(np.fft.rfft(np.roll(np.pad(k, (0, N - len(k))), -(len(k) // 2))))
ax.plot(frec, np.abs(np.fft.rfft(señal)) / N * 2, color=C.grey, lw=0.8,
        label="espectro de la señal")
ax.plot(frec, H, color=C.blue, lw=2.0, label="$|H(f)|$: respuesta del filtro")
ax.set_xlim(0, 12), ax.set_xlabel("frecuencia (Hz)")
ax.set_title("En frecuencia, el filtro es una multiplicación", fontsize=10)
ax.legend(fontsize=7.6)

for n in [1_000, 10_000, 1_000_000]:
    print(f"N={n:>9,}:  directo N^2 = {n**2:.1e},  FFT N log2 N = "
          f"{n*np.log2(n):.1e},  factor {n / np.log2(n):.0f}")
plt.show()

**¿Ocurrió lo que esperabas? ¿Por qué?**

> 

**Juega:** cambia un parámetro de la celda anterior, vuelve a ejecutarla y anota el efecto.

> 


---

## ¿Por qué no se estima un espectro con el módulo de la FFT a secas?

Comparación entre el periodograma crudo y la estimación de Welch para ruido
coloreado más dos tonos.

La figura responde: ¿por qué el periodograma no mejora al tomar más datos?

Ejecutar:  python fig_psd.py

*(script original: `codigo/fig_psd.py`)*

**Antes de ejecutar, escribe aquí qué esperas ver:**

> 


In [ ]:
import pathlib
import sys

import matplotlib.pyplot as plt
import numpy as np
from scipy import signal

from estilo_libro import C, rng, save, use_style  # noqa: E402

use_style()
r = rng(19)

FS = 1000.0
N = 2**16
t = np.arange(N) / FS

# Ruido 1/f más dos tonos débiles
blanco = r.standard_normal(N)
espectro = np.fft.rfft(blanco)
frec = np.fft.rfftfreq(N, 1 / FS)
espectro[1:] /= np.sqrt(frec[1:])
x = np.fft.irfft(espectro, N)
x = x / x.std()
x += 0.35 * np.sin(2 * np.pi * 120 * t) + 0.25 * np.sin(2 * np.pi * 250 * t)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(10.4, 4.2))

# --- Periodograma crudo: no converge -------------------------------------
for n_usar, color, alpha in [(2**12, C.grey, 0.5), (2**16, C.red, 0.8)]:
    f, P = signal.periodogram(x[:n_usar], FS)
    ax1.loglog(f[1:], P[1:], color=color, lw=0.5, alpha=alpha,
               label=f"periodograma, N = {n_usar}")
ax1.set_xlabel("frecuencia (Hz)"), ax1.set_ylabel("densidad espectral")
ax1.set_title("Periodograma: más datos, igual de ruidoso")
ax1.legend(fontsize=8)
ax1.set_ylim(1e-9, 1e1)

# --- Welch: promedia segmentos -------------------------------------------
for nperseg, color in [(256, C.ochre), (1024, C.green), (4096, C.blue)]:
    f, P = signal.welch(x, FS, nperseg=nperseg)
    ax2.loglog(f[1:], P[1:], color=color, lw=1.4,
               label=f"Welch, ventana {nperseg}")
ax2.loglog(f[1:], 2e-3 / f[1:], "--", color=C.ink, lw=1.2, label="$1/f$")
for f0 in (120, 250):
    ax2.axvline(f0, color=C.red, lw=0.8, alpha=0.5)
ax2.set_xlabel("frecuencia (Hz)"), ax2.set_ylabel("densidad espectral")
ax2.set_title("Welch: promediar reduce la varianza")
ax2.legend(fontsize=8)
ax2.set_ylim(1e-9, 1e1)

f1, P1 = signal.periodogram(x, FS)
f2, P2 = signal.welch(x, FS, nperseg=1024)
print(f"desv. típica de log10(P) — periodograma: {np.std(np.log10(P1[10:])):.3f}")
print(f"desv. típica de log10(P) — Welch(1024): {np.std(np.log10(P2[5:])):.3f}")
plt.show()

**¿Ocurrió lo que esperabas? ¿Por qué?**

> 

**Juega:** cambia un parámetro de la celda anterior, vuelve a ejecutarla y anota el efecto.

> 


---

## ¿De verdad se puede hacer una esquina sumando ondas suaves?

Construcción de una onda cuadrada y de un diente de sierra sumando armónicos,
con el fenómeno de Gibbs medido.

La figura responde: ¿converge la serie de Fourier en las discontinuidades?

Ejecutar:  python fig_series_fourier.py

*(script original: `codigo/fig_series_fourier.py`)*

**Antes de ejecutar, escribe aquí qué esperas ver:**

> 


In [ ]:
import pathlib
import sys

import matplotlib.pyplot as plt
import numpy as np

from estilo_libro import C, save, use_style  # noqa: E402

use_style()

x = np.linspace(-np.pi, np.pi, 4000)


def cuadrada_parcial(x, n_arm):
    s = np.zeros_like(x)
    for k in range(1, n_arm + 1, 2):
        s += (4 / (np.pi * k)) * np.sin(k * x)
    return s


fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(10.4, 4.2))

ax1.plot(x, np.sign(np.sin(x)), color=C.ink, lw=2.0, label="onda cuadrada")
for n, color in zip([1, 3, 9, 51], [C.grey, C.ochre, C.green, C.blue]):
    ax1.plot(x, cuadrada_parcial(x, n), color=color, lw=1.3,
             label=f"{(n+1)//2} armónicos")
ax1.set_xlabel("$x$"), ax1.set_ylabel("$f(x)$")
ax1.set_title("Sumando senos impares")
ax1.legend(fontsize=7.6, loc="lower right")
ax1.set_ylim(-1.5, 1.5)

# --- Gibbs -----------------------------------------------------------------
for n, color in zip([11, 51, 201, 1001], [C.ochre, C.green, C.blue, C.red]):
    y = cuadrada_parcial(x, n)
    sobrepaso = y.max()
    ax2.plot(x, y, color=color, lw=1.2,
             label=f"{(n+1)//2} arm.: máx = {sobrepaso:.4f}")
ax2.axhline(1, color=C.ink, lw=1.2)
ax2.axhline(1.17898, color=C.grey, ls="--", lw=1.2)
ax2.text(0.35, 1.19, "límite de Gibbs: 1,17898…", fontsize=8.2, color=C.grey)
ax2.set_xlim(-0.05, 0.6), ax2.set_ylim(0.8, 1.28)
ax2.set_xlabel("$x$ (zoom en la discontinuidad)")
ax2.set_title("El sobrepaso no desaparece: se estrecha")
ax2.legend(fontsize=7.6, loc="lower right")

for n in [11, 51, 201, 1001, 5001]:
    print(f"{(n+1)//2:5d} armónicos -> sobrepaso máximo = "
          f"{cuadrada_parcial(np.linspace(0, 0.5, 20000), n).max():.6f}")
plt.show()

**¿Ocurrió lo que esperabas? ¿Por qué?**

> 

**Juega:** cambia un parámetro de la celda anterior, vuelve a ejecutarla y anota el efecto.

> 
